# Stage B validation — summary

Bundles the two endpoints of the validation story:
- **§8.2.1** internal-consistency *gate* — run **before** AERONET-blind tests to reject overfit/underfit candidates.
- **§8.2.6** success-criteria *scorecard* — the headline baseline / target / achieved table.

Prerequisites:
- Trained RF bundle at `MODELS_DIR/rf_primary.joblib`.
- B3 RF gap-filled per-slot NetCDFs in `RF_OUTPUT_DIR`.

If missing, run `python run_stage_b.py all --start 2022-09-01 --end 2026-04-30` first.

In [ ]:
from datetime import date
import sys
from pathlib import Path
import numpy as np
import pandas as pd

sys.path.insert(0, str(Path.cwd()))
import validate as vb
import rf_gapfill as rf
import config as cfg

START = cfg.TEST_START
END   = cfg.TEST_END
print(f'Held-out window: {START} → {END}')

## §8.2.1 — Internal consistency gate

Reads the train / CV / internal-test RMSEs stored on the trained RF bundle.  Checks are **one-sided** — train and test only fail when *worse* than CV by the tolerance margin.  Train *better* than CV is expected (the final fit uses every training day with no held-out block); a calmer chronological test slice may also legitimately beat the worst CV fold.

Pass criteria:
- `train_underfit_pass` — `rmse_train ≤ rmse_cv_mean · (1+tol)`.  Fails only if the model can't fit training data (true underfit signal).
- `test_overfit_pass` — `rmse_internal_test ≤ rmse_cv_mean · 1.3`.  Fails only when the held-out test is catastrophically worse than CV.

Also reports `rmse_cv_std`, `rmse_cv_max`, `worst_fold_idx` so a high `rmse_cv_mean` driven by one seasonal-hot-spot fold is visible.  Inspect `bundle.cv_fold_metrics[worst_fold_idx]` to see which time block was hardest.

In [ ]:
bundle = rf.load_bundle("rf_residual_stctx_tau6")
print('Training window:', bundle.training_window)
print('Hyperparameters:', bundle.hyperparams)
print('Metrics       :', bundle.metrics)

diag = vb.internal_consistency(bundle.metrics, bundle.cv_fold_metrics)
pd.DataFrame([diag])

## §8.2.6 — Success-criteria scorecard

Headline §9 table.  Achieved values come from the **full** RF product (`blind_only=False`) and the post-fill coverage audit.

In [ ]:
pairs_full = vb.aeronet_pairs(START, END, candidate='rf', blind_only=False)
cov        = vb.coverage_audit(START, END)
vb.success_table(pairs_full, cov)

In [ ]:
pairs_full_kg = vb.aeronet_pairs(START, END, candidate='kriging', blind_only=False)
cov_kg        = vb.coverage_audit(START, END, candidate='kriging')
vb.success_table(pairs_full_kg, cov_kg)